# talon.bridge Tutorial: Export & Import

Bidirectional PyTorch <-> TALON IR conversion.

In [ ]:
import torch, torch.nn as nn, numpy as np
from talon import bridge, ir

## 1. Standard Export

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = Net()
sample = torch.randn(1, 784)
graph = bridge.to_ir(model, sample)
for n, nd in graph.nodes.items():
    print(f"  {n}: {type(nd).__name__}")

## 2. Round-Trip

In [ ]:
ex = bridge.ir_to_torch(graph)
with torch.no_grad():
    d = (model(sample) - ex(sample)).abs().max().item()
print(f"Diff: {d:.2e}, Eq: {d < 1e-5}")

## 3. Spiking Export

In [ ]:
gs = bridge.to_ir(model, sample, spiking=True, weight_bits=8)
for n, nd in gs.nodes.items():
    print(f"  {n}: {type(nd).__name__}")

## 4. Stateful LIF

In [ ]:
nodes = {
    "input": ir.Input(np.array([128])),
    "fc": ir.Affine(weight=np.random.randn(64, 128).astype(np.float32)*0.05, bias=np.zeros(64, dtype=np.float32)),
    "lif": ir.LIF(tau=np.ones(64, dtype=np.float32)*5, r=np.ones(64, dtype=np.float32), v_leak=np.zeros(64, dtype=np.float32), v_threshold=np.ones(64, dtype=np.float32)*0.5),
    "output": ir.Output(np.array([64])),
}
e = [("input","fc"),("fc","lif"),("lif","output")]
exec_lif = bridge.ir_to_torch(ir.Graph(nodes=nodes, edges=e), return_state=True)
x = torch.randn(1, 128)*0.3
state = {}
for t in range(5):
    out, state = exec_lif(x, state)
    print(f"  t={t}: spikes={(out>0).sum().item()}/64")

## 5. CyclicGraphExecutor

In [ ]:
mods = {
    "fc1": nn.Linear(64, 32),
    "lif1": bridge.LIFModule(tau=np.ones(32, dtype=np.float32)*10, r=np.ones(32, dtype=np.float32), v_leak=np.zeros(32, dtype=np.float32), v_threshold=np.ones(32, dtype=np.float32)),
    "fc_fb": nn.Linear(32, 64),
}
edges = [("input","fc1"),("fc1","lif1"),("lif1","output"),("lif1","fc_fb"),("fc_fb","fc1")]
ec = bridge.CyclicGraphExecutor(modules=mods, edges=edges, timesteps=3, output_nodes=["lif1"], return_state=True)
outs, st = ec(torch.randn(1,64)*0.1, {})
print(f"Timesteps: {len(outs)}")
for t, o in enumerate(outs):
    print(f"  t={t}: {o.shape}")